# 04 · Hiperparametre optimizasyonu (Optuna) — en güçlü 3 model

**Önce GPU açın.** Colab: `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`. Kaggle: sağ panel → *Session options* → *Accelerator* → GPU ve *Internet: On*. Bu notebook her iki platformda da çalışır.

- Adaylar: `results.csv`'de validation macro F1'e göre en iyi 3 model (otomatik seçilir)
- Her model için **25 deneme**; eğitim kümesinin %40'lık sabit-seed alt kümesi ve kısaltılmış epoch bütçesi; MedianPruner ile budama
- Amaç: validation macro F1. **Test kümesi yüklenmez.**
- Bittikten sonra en iyi parametrelerle model **tam eğitim kümesinde** yeniden eğitilir (`<model>__opt`)
- Oturum kopsa bile aynı notebook'u yeniden çalıştırmanız yeterli: biten denemeler Drive'dan okunur, kalınan yerden devam edilir

⏱ Model başına yaklaşık 40–60 dk (toplam 2–3 saat). İstediğiniz modeli ayrı oturumda da çalıştırabilirsiniz.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 optuna==5.0.0
import torch
assert torch.cuda.is_available(), 'GPU yok! Colab: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU | Kaggle: sağ panel Session options > Accelerator > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## Adayların seçimi

In [ ]:
from src import optimize as O
import pandas as pd
CANDIDATES = O.top_candidates(3)
print('Optuna adayları (val macro F1 sırasıyla):', CANDIDATES)
r = pd.read_csv(C.OUTPUT_DIR / 'results.csv')
display(r[r.model_name.isin(CANDIDATES)][['model_name','val_macro_f1','val_balanced_accuracy','val_qwk']].round(4))

## Arama + yeniden eğitim

Her aday için sırayla çalışır.

In [ ]:
from IPython.display import Image, display
for key in CANDIDATES:
    O.run_search(key, n_trials=25, subset_fraction=0.4)
    O.retrain_best(key)
    display(Image(filename=str(C.FIG_DIR / f'optuna_{key}.png'), width=900))

## Özet: ayarsız ve Optuna ile ayarlı (validation)

In [ ]:
import json
r = pd.read_csv(C.OUTPUT_DIR / 'results.csv')
rows = []
for k in CANDIDATES:
    a = r[r.model_name == k].iloc[0]; b = r[r.model_name == f'{k}__opt'].iloc[0]
    s = json.load(open(C.OUTPUT_DIR / 'optuna' / k / 'search_summary.json'))
    rows.append({'model': k, 'F1_ayarsiz': a.val_macro_f1, 'F1_optuna': b.val_macro_f1,
                 'fark': b.val_macro_f1 - a.val_macro_f1, 'QWK_ayarsiz': a.val_qwk, 'QWK_optuna': b.val_qwk,
                 'denemeler': s['trial_states']})
display(pd.DataFrame(rows).round(4))
for k in CANDIDATES:
    print(k, json.load(open(C.OUTPUT_DIR / 'optuna' / k / 'search_summary.json'))['best_params'])

`outputs/results.csv` ve `outputs/optuna/` klasöründeki `search_summary.json` dosyalarını paylaşın.